In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import os

from sklearn.metrics import mean_absolute_error, mean_squared_error

# ---------------------------
# OUTPUT FOLDERS
# ---------------------------
OUTPUT_DIR = "outputs"
LOSS_DIR = os.path.join(OUTPUT_DIR, "loss")
PRED_DIR = os.path.join(OUTPUT_DIR, "predictions")

os.makedirs(LOSS_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

# ---------------------------
# PARAMETERS (MANDATORY)
# ---------------------------
window_size = 10 #(sum of all digits) mod 10 + 8 = 12mod10 +8
prediction_horizon = 1 #(last 2 digits) mod 3 + 1 = 00mod3 + 1
hidden_size = 14 #(first 3 digits) mod 16 + 8 = 102mod16 + 8 = 6+8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------
# WINDOW FUNCTION
# ---------------------------
def create_windows(data, window_size, horizon):
    X, Y = [], []

    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size])
        Y.append(data[i+window_size:i+window_size+horizon])

    # WHY:
    # Converts sequential time-series into supervised learning format
    return np.array(X), np.array(Y)

# ---------------------------
# MODELS
# ---------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(window_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, prediction_horizon)
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        # WHY:
        # MLP treats inputs independently → requires flattening
        return self.model(x)

class CustomRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.Wx = nn.Linear(1, hidden_size)
        self.Wh = nn.Linear(hidden_size, hidden_size)
        self.fc = nn.Linear(hidden_size, prediction_horizon)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # WHY:
        # Initialize hidden state to zero → no prior information
        h = torch.zeros(batch_size, hidden_size).to(x.device)

        for t in range(seq_len):
            xt = x[:, t, :]
            h = torch.tanh(self.Wx(xt) + self.Wh(h))

        # WHY:
        # Hidden state carries temporal memory across timesteps
        return self.fc(h)

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, prediction_horizon)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_proj = nn.Linear(1, 16)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=16, nhead=2, batch_first=True),
            num_layers=1
        )

        self.fc = nn.Linear(16 * window_size, prediction_horizon)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)
        x = x.reshape(x.size(0), -1)
        return self.fc(x)

# ---------------------------
# TRAINING
# ---------------------------
def train_model(model, X, Y, epochs=50):
    model.to(device)
    X, Y = X.to(device), Y.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    losses = []

    for epoch in range(epochs):
        preds = model(X)
        loss = criterion(preds, Y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return losses

# ---------------------------
# EVALUATION
# ---------------------------
def evaluate(model, X, Y):
    model.eval()
    X, Y = X.to(device), Y.to(device)

    with torch.no_grad():
        preds = model(X).cpu().numpy()
        Y = Y.cpu().numpy()

    mae = mean_absolute_error(Y, preds)
    mse = mean_squared_error(Y, preds)
    rmse = np.sqrt(mse)

    return preds, mae, mse, rmse

# ---------------------------
# PIPELINE
# ---------------------------
def run_pipeline(data, dataset_name):
    print(f"\n--- Running on {dataset_name} ---")

    # WHY:
    # Normalize data to stabilize training and avoid scale dominance
    data = (data - np.mean(data)) / np.std(data)

    X, Y = create_windows(data, window_size, prediction_horizon)

    split = int(0.8 * len(X))

    # WHY:
    # Chronological split prevents future data leakage
    X_train, X_test = X[:split], X[split:]
    Y_train, Y_test = Y[:split], Y[split:]

    X_train = torch.tensor(X_train, dtype=torch.float32)
    Y_train = torch.tensor(Y_train, dtype=torch.float32)
    X_test = torch.tensor(X_test, dtype=torch.float32)
    Y_test = torch.tensor(Y_test, dtype=torch.float32)

    X_train_rnn = X_train.unsqueeze(-1)
    X_test_rnn = X_test.unsqueeze(-1)

    models = {
        "MLP": MLP(),
        "RNN": CustomRNN(),
        "LSTM": LSTMModel(),
        "Transformer": TransformerModel()
    }

    for model_name, model in models.items():
        print(f"\nTraining {model_name}...")

        if model_name == "MLP":
            loss = train_model(model, X_train, Y_train)
            preds, mae, mse, rmse = evaluate(model, X_test, Y_test)
        else:
            loss = train_model(model, X_train_rnn, Y_train)
            preds, mae, mse, rmse = evaluate(model, X_test_rnn, Y_test)

        print(f"{model_name} → MAE: {mae:.4f}, RMSE: {rmse:.4f}")

        plt.figure()
        plt.plot(loss)
        plt.title(f"{model_name} Loss - {dataset_name}")
        plt.savefig(os.path.join(LOSS_DIR, f"{dataset_name}_{model_name}_loss.png"))
        plt.close()

        plt.figure()
        plt.plot(Y_test.numpy(), label="Actual")
        plt.plot(preds, label="Predicted")
        plt.legend()
        plt.title(f"{model_name} Prediction - {dataset_name}")
        plt.savefig(os.path.join(PRED_DIR, f"{dataset_name}_{model_name}_prediction.png"))
        plt.close()

# ---------------------------
# ABLATION STUDY
# ---------------------------
def ablation(data, dataset_name):
    print(f"\n--- Ablation Study: {dataset_name} ---")

    data = (data - np.mean(data)) / np.std(data)

    for w in [window_size//2, window_size, window_size*2]:
        print(f"\nWindow Size: {w}")

        X, Y = create_windows(data, w, prediction_horizon)

        split = int(0.8 * len(X))

        X_train = torch.tensor(X[:split], dtype=torch.float32).unsqueeze(-1)
        Y_train = torch.tensor(Y[:split], dtype=torch.float32)

        X_test = torch.tensor(X[split:], dtype=torch.float32).unsqueeze(-1)
        Y_test = torch.tensor(Y[split:], dtype=torch.float32)

        model = CustomRNN()
        train_model(model, X_train, Y_train, epochs=30)

        _, mae, _, rmse = evaluate(model, X_test, Y_test)

        print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}")

# ---------------------------
# MAIN
# ---------------------------
if __name__ == "__main__":

    # DATASET 1
    df1 = pd.read_csv("Electric_Production.csv")
    data1 = df1["Value"].values.astype(float)

    run_pipeline(data1, "Electric_Production")
    ablation(data1, "Electric_Production")

    # DATASET 2
    df2 = pd.read_csv("AirQuality.csv", sep=';', decimal=',')

    data2 = pd.to_numeric(df2["CO(GT)"], errors='coerce')
    data2 = data2.replace(-200, np.nan).dropna().values.astype(float)

    run_pipeline(data2, "AirQuality")
    ablation(data2, "AirQuality")


--- Running on Electric_Production ---

Training MLP...
MLP → MAE: 0.7582, RMSE: 0.9626

Training RNN...
RNN → MAE: 0.6715, RMSE: 0.8382

Training LSTM...
LSTM → MAE: 0.7737, RMSE: 0.9796

Training Transformer...
Transformer → MAE: 0.2379, RMSE: 0.3107

--- Ablation Study: Electric_Production ---

Window Size: 5
MAE: 1.2262, RMSE: 1.4111

Window Size: 10
MAE: 0.9978, RMSE: 1.1929

Window Size: 20
MAE: 0.6137, RMSE: 0.7722

--- Running on AirQuality ---

Training MLP...
MLP → MAE: 0.6177, RMSE: 0.7980

Training RNN...
RNN → MAE: 0.4261, RMSE: 0.6059

Training LSTM...
LSTM → MAE: 0.6456, RMSE: 0.8180

Training Transformer...
Transformer → MAE: 0.3622, RMSE: 0.5217

--- Ablation Study: AirQuality ---

Window Size: 5
MAE: 0.7380, RMSE: 0.9268

Window Size: 10
MAE: 0.4883, RMSE: 0.6426

Window Size: 20
MAE: 0.5764, RMSE: 0.7724
